01 — OOF - Jaime.

Feature Engineering (Variables Clave)
- log_last_listing_price: Es la variable con mayor correlación; se recuperaron los nulos usando el valor catastral (taxAssessedValue) multiplicado por un ratio mediano para no perder datos.
- log_taxAssessedValue: El valor catastral funciona como un excelente predictor del precio de venta final.
- Geo-clustering (KMeans): Se crearon 30 clusters basados en latitud y longitud para capturar micro-mercados que el código postal ignora.
- Target Encoding: Se incorporó el precio mediano histórico por zona (zipcode y clusters) directamente como característica.
- Null Flags: Se crearon indicadores binarios para campos vacíos (como last_listing_price_is_null), ya que la ausencia de datos también aporta información.

Optimización (Optuna)
- Sintonización de Hiperparámetros: Se realizaron 80 pruebas optimizando directamente el MAPE, asegurando que el modelo esté alineado con la métrica de la competencia.
- Control de Overfitting: Se implementó early stopping en cada fold para evitar que el modelo se memorice el set de entrenamiento.

Estrategia de Ensamble (Blending)
- LGBM + CatBoost: La combinación de ambos modelos aporta diversidad, ya que cada uno procesa variables categóricas y nulos de forma distinta.
- Pesos Óptimos: El peso de cada modelo en el ensamble final se define mediante una búsqueda en grilla sobre tus resultados OOF.

Ejecutar desde el directorio participant/scripts/02_lgbm_basic.ipynb

    participant/
    ├── submissions/ (RESULTADOS)
    ├── scripts/ (DONDE TIENE QUE ESTAR ESTE ARCHIVO)
    ├── participant.zip/

Qué se agrega sobre la versión anterior:

- Target encoding con K-Fold real (sin leakage)
- Distancias a 5 ciudades clave de Florida
- Ratio listing/tax por zipcode en vez de global
- Winsorización del target (percentil 1-99)
- XGBoost tuneado con Optuna
- Búsqueda de pesos del blend en grilla 3D

In [ ]:

# INSTALACIÓN
!pip install optuna lightgbm catboost xgboost --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.2 MB/s eta 0:00:00


In [ ]:
# IMPORTS
import os
import zipfile
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import catboost as cb
import xgboost as xgb
import optuna
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# GOOGLE DRIVE Y RUTAS
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = "/content/drive/MyDrive/participant/participant.zip"
SUBMISSIONS_DRIVE_DIR = "/content/drive/MyDrive/participant/submissions"
os.makedirs(SUBMISSIONS_DRIVE_DIR, exist_ok=True)

csv_train = csv_test = None
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    for f in zip_ref.namelist():
        if f.endswith("train_processed.csv"): csv_train = f
        if f.endswith("test_processed.csv"):  csv_test  = f
    zip_ref.extract(csv_train, path="/content/")
    zip_ref.extract(csv_test,  path="/content/")
    print(f"Extraídos:\n  {csv_train}\n  {csv_test}")

path_train = os.path.join("/content", csv_train)
path_test  = os.path.join("/content", csv_test)

Mounted at /content/drive
Extraídos:
  participant/data/tabular/train_processed.csv
  participant/data/tabular/test_processed.csv


In [ ]:
# CARGA DE DATOS
train_raw = pd.read_csv(path_train)
test_raw  = pd.read_csv(path_test)

TARGET = "log_price"
print(f"Train: {train_raw.shape} | Test: {test_raw.shape}")

In [ ]:
# FEATURE ENGINEERING
def feature_engineering(train_df, test_df):
    train_df = train_df.copy()
    test_df  = test_df.copy()
    n_train  = len(train_df)
    combined = pd.concat([train_df, test_df], axis=0, ignore_index=True)

    # Flags de nulos
    null_flag_cols = [
        'lotAreaValue', 'last_listing_price', 'bath_to_bed_ratio',
        'taxAssessedValue', 'yearBuilt', 'bedrooms', 'bathrooms'
    ]
    for col in null_flag_cols:
        combined[f'{col}_is_null'] = combined[col].isnull().astype(int)

    # Imputación con medianas de train
    medians = train_df[[
        'bedrooms', 'bathrooms', 'livingArea', 'yearBuilt',
        'lotAreaValue', 'taxAssessedValue', 'latest_tax_value',
        'latest_tax_paid', 'bath_to_bed_ratio'
    ]].median()
    for col in medians.index:
        combined[col] = combined[col].fillna(medians[col])

    combined['latitude']  = combined['latitude'].fillna(combined['latitude'].median())
    combined['longitude'] = combined['longitude'].fillna(combined['longitude'].median())

    combined['log_living_area'] = np.log1p(combined['livingArea'].clip(lower=0))
    combined['lotAreaValue']    = combined['lotAreaValue'].fillna(0)
    combined['log_lot_area']    = np.log1p(combined['lotAreaValue'].clip(lower=0))
    combined['property_age']    = combined['property_age'].fillna(2024 - combined['yearBuilt'])

    # Imputación inteligente de last_listing_price
    mask_train_both = (
        train_df['last_listing_price'].notnull() &
        train_df['taxAssessedValue'].notnull()
    )
    ratio_list_to_tax = (
        train_df.loc[mask_train_both, 'last_listing_price'] /
        train_df.loc[mask_train_both, 'taxAssessedValue'].replace(0, np.nan)
    ).median()

    llp_null = combined['last_listing_price'].isnull()
    combined.loc[llp_null, 'last_listing_price'] = (
        combined.loc[llp_null, 'taxAssessedValue'] * ratio_list_to_tax
    )
    combined['last_listing_price'] = combined['last_listing_price'].fillna(
        train_df['last_listing_price'].median()
    )

    # Geo-clustering
    N_CLUSTERS = 30
    coords = combined[['latitude', 'longitude']].values
    km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
    combined['geo_cluster'] = km.fit_predict(coords)

    # Distancias a ciudades clave de Florida (en grados, proxy de km)
    CITIES = {
        'miami':       (25.7617, -80.1918),
        'orlando':     (28.5383, -81.3792),
        'tampa':       (27.9506, -82.4572),
        'jacksonville':(30.3322, -81.6557),
        'naples':      (26.1420, -81.7948),
    }
    for city, (clat, clon) in CITIES.items():
        combined[f'dist_{city}'] = np.sqrt(
            (combined['latitude']  - clat) ** 2 +
            (combined['longitude'] - clon) ** 2
        )
    combined['dist_nearest_city'] = combined[
        [f'dist_{c}' for c in CITIES]
    ].min(axis=1)

    # Target encoding con K-Fold para evitar leakage
    # Solo se calcula sobre train, test recibe el encoding de train completo
    train_part = combined.iloc[:n_train].copy()
    test_part  = combined.iloc[n_train:].copy()

    kf_enc = KFold(n_splits=5, shuffle=True, random_state=42)
    for enc_col in ['zipcode', 'zip_3digit', 'geo_cluster']:
        train_part[f'{enc_col}_target_enc'] = np.nan
        global_median = train_part[TARGET].median()

        for tr_idx, val_idx in kf_enc.split(train_part):
            fold_map = (
                train_part.iloc[tr_idx]
                .groupby(enc_col)[TARGET]
                .median()
            )
            train_part.loc[
                train_part.index[val_idx], f'{enc_col}_target_enc'
            ] = train_part.iloc[val_idx][enc_col].map(fold_map).fillna(global_median).values

        # Para test: encoding con todo el train
        full_map = train_part.groupby(enc_col)[TARGET].median()
        test_part[f'{enc_col}_target_enc'] = (
            test_part[enc_col].map(full_map).fillna(global_median)
        )

    combined = pd.concat([train_part, test_part], axis=0, ignore_index=True)

    # Ratio last_listing_price / taxAssessedValue por zipcode
    zip_ratio_map = (
        combined.iloc[:n_train]
        .groupby('zipcode')
        .apply(lambda g: (
            g['last_listing_price'] /
            g['taxAssessedValue'].replace(0, np.nan)
        ).median())
    )
    combined['zip_list_tax_ratio'] = (
        combined['zipcode'].map(zip_ratio_map).fillna(ratio_list_to_tax)
    )

    # Nuevas features numéricas
    combined['log_last_listing_price'] = np.log1p(
        combined['last_listing_price'].clip(lower=0)
    )
    combined['log_taxAssessedValue'] = np.log1p(
        combined['taxAssessedValue'].clip(lower=0)
    )
    combined['list_to_tax_ratio'] = (
        combined['last_listing_price'] /
        combined['taxAssessedValue'].replace(0, np.nan)
    ).fillna(ratio_list_to_tax)

    combined['area_per_bedroom'] = (
        combined['livingArea'] / combined['bedrooms'].replace(0, np.nan)
    ).fillna(combined['livingArea'])

    combined['tax_per_sqft'] = (
        combined['taxAssessedValue'] / combined['livingArea'].replace(0, np.nan)
    ).fillna(0)

    combined['hoa_annual'] = combined['hoa_fee_monthly'] * 12

    combined['premium_property'] = (
        combined['has_waterfront'].astype(int) +
        combined['has_pool'].astype(int) +
        combined['has_garage'].astype(int)
    )
    combined['distress_score'] = (
        combined['tag_foreclosure'].astype(int) +
        combined['tag_price_cut'].astype(int) -
        combined['tag_new_construction'].astype(int)
    )

    combined['decade_built'] = (combined['yearBuilt'] // 10 * 10).astype(int)

    combined['bath_per_sqft'] = (
        combined['bathrooms'] / combined['livingArea'].replace(0, np.nan)
    ).fillna(0)

    combined['desc_density'] = (
        combined['desc_word_count'] / combined['desc_length'].replace(0, np.nan)
    ).fillna(0)

    combined['school_score'] = (
        combined['avg_school_rating'] * combined['num_nearby_schools'] /
        (combined['min_school_distance'] + 0.1)
    )

    # Winsorización del target (solo en train, no afecta test)
    # Se hace afuera, sobre y_all, no acá

    # Recorta los precios extremos (outliers) en los percentiles 1 y 99 para que los modelos no se distraigan con mansiones excesivamente caras o ventas simbólicas de $1.

    # Encodear categoricals
    for col in ['homeType', 'zipcode', 'zip_3digit']:
        combined[col] = combined[col].astype('category')
    combined['geo_cluster']  = combined['geo_cluster'].astype('category')
    combined['decade_built'] = combined['decade_built'].astype('category')

    train_fe = combined.iloc[:n_train].reset_index(drop=True)
    test_fe  = combined.iloc[n_train:].reset_index(drop=True)

    return train_fe, test_fe, ratio_list_to_tax


print("Aplicando feature engineering...")
train, test, ratio_list_to_tax = feature_engineering(train_raw, test_raw)
print(f"Columnas totales: {train.shape[1]}")


Aplicando feature engineering...
Columnas totales: 76


In [ ]:
# DEFINICIÓN DE FEATURES
FEATURES = [
    # Core
    "bedrooms", "bathrooms", "livingArea", "yearBuilt",
    "latitude", "longitude", "lotAreaValue", "photoCount",
    "homeType", "zipcode", "zip_3digit",
    # Tax & Financial
    "taxAssessedValue", "propertyTaxRate", "latest_tax_value",
    "latest_tax_paid", "num_tax_records", "last_listing_price",
    # Transaction
    "num_sales", "num_price_changes",
    # Schools
    "avg_school_rating", "max_school_rating", "num_nearby_schools",
    "min_school_distance",
    # Attributes
    "has_hoa", "hoa_fee_monthly", "has_pool", "has_garage", "has_waterfront",
    # Tags
    "tag_price_cut", "tag_new_construction", "tag_foreclosure",
    # Derived originales
    "property_age", "bath_to_bed_ratio", "log_living_area", "log_lot_area",
    # Description
    "desc_length", "desc_word_count", "desc_is_boilerplate",
    "desc_mentions_renovated", "desc_mentions_pool",
    "desc_mentions_view", "desc_mentions_new",
    # Geo
    "geo_cluster",
    "dist_miami", "dist_orlando", "dist_tampa",
    "dist_jacksonville", "dist_naples", "dist_nearest_city",
    # Target encodings (K-Fold, sin leakage)
    "zipcode_target_enc", "zip_3digit_target_enc", "geo_cluster_target_enc",
    # Ratios y logs
    "log_last_listing_price", "log_taxAssessedValue",
    "list_to_tax_ratio", "zip_list_tax_ratio",
    "area_per_bedroom", "tax_per_sqft",
    # Composites
    "hoa_annual", "premium_property", "distress_score",
    "decade_built", "bath_per_sqft", "desc_density", "school_score",
    # Null flags
    "lotAreaValue_is_null", "last_listing_price_is_null",
    "taxAssessedValue_is_null", "yearBuilt_is_null",
]

CAT_FEATURES = [
    "homeType", "zipcode", "zip_3digit", "geo_cluster", "decade_built"
]

missing = [f for f in FEATURES if f not in train.columns]
if missing:
    print(f"Features faltantes: {missing}")
else:
    print(f"Todas las {len(FEATURES)} features disponibles")

# Winsorización del target
y_raw = train[TARGET]
lower = y_raw.quantile(0.01)
upper = y_raw.quantile(0.99)
y_all = y_raw.clip(lower, upper)
X_all = train[FEATURES]

print(f"Target winsorizados: [{lower:.3f}, {upper:.3f}]")
print(f"Filas afectadas: {((y_raw < lower) | (y_raw > upper)).sum()}")

✅ Todas las 69 features disponibles
Target winsorizados: [11.323, 14.305]
Filas afectadas: 238


In [ ]:
# MÉTRICAS
def mape(y_true_log, y_pred_log):
    y_true = np.expm1(np.array(y_true_log))
    y_pred = np.expm1(np.array(y_pred_log))
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def mean_roi(y_true_log, y_pred_log):
    y_true = np.expm1(np.array(y_true_log))
    y_pred = np.expm1(np.array(y_pred_log))
    return np.mean(np.abs(y_pred - y_true) / y_true) * 100


In [ ]:
# CONFIG GENERAL
N_SPLITS    = 5
RANDOM_SEED = 42
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

In [ ]:
# OPTUNA — LIGHTGBM
N_TRIALS_LGBM = 40   # bajá a 40 si querés ir rápido

def objective_lgbm(trial):
    params = {
        "objective":           "regression",
        "metric":              "mae",
        "verbosity":           -1,
        "boosting_type":       "gbdt",
        "random_state":        RANDOM_SEED,
        "n_estimators":        trial.suggest_int("n_estimators", 300, 2000),
        "learning_rate":       trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "num_leaves":          trial.suggest_int("num_leaves", 20, 300),
        "max_depth":           trial.suggest_int("max_depth", 3, 12),
        "min_child_samples":   trial.suggest_int("min_child_samples", 10, 100),
        "feature_fraction":    trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction":    trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq":        trial.suggest_int("bagging_freq", 1, 7),
        "lambda_l1":           trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2":           trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "min_gain_to_split":   trial.suggest_float("min_gain_to_split", 0.0, 1.0),
    }
    fold_mapes = []
    for tr_idx, val_idx in kf.split(X_all):
        X_tr,  X_val = X_all.iloc[tr_idx], X_all.iloc[val_idx]
        y_tr,  y_val = y_all.iloc[tr_idx], y_all.iloc[val_idx]
        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            categorical_feature=CAT_FEATURES,
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(period=-1)
            ]
        )
        fold_mapes.append(mape(y_val.values, model.predict(X_val)))
    return np.mean(fold_mapes)

print(f"Optuna LGBM — {N_TRIALS_LGBM} trials...")
study_lgbm = optuna.create_study(direction="minimize", study_name="lgbm_mape")
study_lgbm.optimize(objective_lgbm, n_trials=N_TRIALS_LGBM, show_progress_bar=True)
print(f"Mejor MAPE LGBM: {study_lgbm.best_value:.3f}%")

🔍 Optuna LGBM — 40 trials...


  0%|          | 0/40 [00:00<?, ?it/s]

✅ Mejor MAPE LGBM: 25.354%


In [ ]:
# OPTUNA — CATBOOST
N_TRIALS_CB = 40   # CatBoost es más lento por trial

# Para Optuna, CatBoost necesita los categoricals como string
X_cb = train[FEATURES].copy()
for col in CAT_FEATURES:
    X_cb[col] = X_cb[col].astype(str)
cat_indices_cb = [FEATURES.index(c) for c in CAT_FEATURES]

def objective_cb(trial):
    params = {
        "loss_function":       "MAE",
        "eval_metric":         "MAE",
        "random_seed":         RANDOM_SEED,
        "verbose":             False,
        "od_type":             "Iter",
        "od_wait":             50,
        "iterations":          trial.suggest_int("iterations", 300, 1500),
        "learning_rate":       trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "depth":               trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg":         trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength":     trial.suggest_float("random_strength", 0.0, 2.0),
        "border_count":        trial.suggest_int("border_count", 32, 255),
    }
    fold_mapes = []
    for tr_idx, val_idx in kf.split(X_cb):
        X_tr  = X_cb.iloc[tr_idx]
        X_val = X_cb.iloc[val_idx]
        y_tr  = y_all.iloc[tr_idx]
        y_val = y_all.iloc[val_idx]

        dtrain = cb.Pool(X_tr,  y_tr,  cat_features=cat_indices_cb)
        dval   = cb.Pool(X_val, y_val, cat_features=cat_indices_cb)

        model_cb = cb.CatBoostRegressor(**params)
        model_cb.fit(dtrain, eval_set=dval, early_stopping_rounds=50, verbose=False)
        fold_mapes.append(mape(y_val.values, model_cb.predict(X_val)))
    return np.mean(fold_mapes)

print(f"\nOptuna CatBoost — {N_TRIALS_CB} trials...")
study_cb = optuna.create_study(direction="minimize", study_name="cb_mape")
study_cb.optimize(objective_cb, n_trials=N_TRIALS_CB, show_progress_bar=True)
print(f"Mejor MAPE CatBoost: {study_cb.best_value:.3f}%")



🔍 Optuna CatBoost — 40 trials...


  0%|          | 0/40 [00:00<?, ?it/s]

[W 2026-05-08 00:33:10,176] Trial 29 failed with parameters: {'iterations': 1123, 'learning_rate': 0.020494925142171445, 'depth': 8, 'l2_leaf_reg': 0.01962671615054032, 'bagging_temperature': 0.5943617672426829, 'random_strength': 0.6160479132096628, 'border_count': 122} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_23204/4031257263.py", line 39, in objective_cb
    model_cb.fit(dtrain, eval_set=dval, early_stopping_rounds=50, verbose=False)
  File "/usr/local/lib/python3.12/dist-packages/catboost/core.py", line 6178, in fit
    return self._fit(X, y, cat_features, text_features, embedding_features, None, graph, sample_weight, None, None, None, None, baseline,
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

KeyboardInterrupt: 

In [ ]:
# OPTUNA — XGBOOST
N_TRIALS_XGB = 40

# XGBoost no maneja categoricals nativamente: label encode
X_xgb = train[FEATURES].copy()
from sklearn.preprocessing import LabelEncoder
le_dict = {}
for col in CAT_FEATURES:
    le = LabelEncoder()
    X_xgb[col] = le.fit_transform(X_xgb[col].astype(str))
    le_dict[col] = le

X_xgb_test = test[FEATURES].copy()
for col in CAT_FEATURES:
    X_xgb_test[col] = le_dict[col].transform(
        X_xgb_test[col].astype(str).map(
            lambda x: x if x in le_dict[col].classes_ else le_dict[col].classes_[0]
        )
    )

def objective_xgb(trial):
    params = {
        "objective":        "reg:absoluteerror",
        "eval_metric":      "mae",
        "verbosity":        0,
        "seed":             RANDOM_SEED,
        "n_estimators":     trial.suggest_int("n_estimators", 300, 2000),
        "learning_rate":    trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 50),
        "subsample":        trial.suggest_float("subsample", 0.4, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "tree_method":      "hist",
        "early_stopping_rounds": 50,
    }
    fold_mapes = []
    for tr_idx, val_idx in kf.split(X_xgb):
        X_tr,  X_val = X_xgb.iloc[tr_idx], X_xgb.iloc[val_idx]
        y_tr,  y_val = y_all.iloc[tr_idx],  y_all.iloc[val_idx]

        model_xgb = xgb.XGBRegressor(**params)
        model_xgb.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        fold_mapes.append(mape(y_val.values, model_xgb.predict(X_val)))
    return np.mean(fold_mapes)

print(f"\nOptuna XGBoost — {N_TRIALS_XGB} trials...")
study_xgb = optuna.create_study(direction="minimize", study_name="xgb_mape")
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS_XGB, show_progress_bar=True)
print(f"Mejor MAPE XGBoost: {study_xgb.best_value:.3f}%")




In [ ]:
# OOF FINAL — LIGHTGBM
best_params_lgbm = {
    "objective": "regression", "metric": "mae",
    "verbosity": -1, "random_state": RANDOM_SEED,
    **study_lgbm.best_params
}

oof_lgbm  = np.zeros(len(train))
test_lgbm = np.zeros(len(test))
scores_lgbm = []

print("\nEntrenando LGBM final OOF...")
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_all)):
    X_tr,  X_val = X_all.iloc[tr_idx], X_all.iloc[val_idx]
    y_tr,  y_val = y_all.iloc[tr_idx], y_all.iloc[val_idx]

    m = lgb.LGBMRegressor(**best_params_lgbm)
    m.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        categorical_feature=CAT_FEATURES,
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)]
    )
    oof_lgbm[val_idx] = m.predict(X_val)
    test_lgbm += m.predict(test[FEATURES]) / N_SPLITS
    scores_lgbm.append(mape(y_val.values, oof_lgbm[val_idx]))
    print(f"  Fold {fold+1} MAPE: {scores_lgbm[-1]:.2f}%")

print(f"LGBM OOF MAPE: {np.mean(scores_lgbm):.2f}% ± {np.std(scores_lgbm):.2f}%")


In [ ]:
# OOF FINAL — CATBOOST
best_params_cb = {
    "loss_function": "MAE", "eval_metric": "MAE",
    "random_seed": RANDOM_SEED, "verbose": False,
    "od_type": "Iter", "od_wait": 50,
    **study_cb.best_params
}

X_cb_test = test[FEATURES].copy()
for col in CAT_FEATURES:
    X_cb[col]      = X_cb[col].astype(str)
    X_cb_test[col] = X_cb_test[col].astype(str)

oof_cb   = np.zeros(len(train))
test_cb  = np.zeros(len(test))
scores_cb = []

print("\n📊 Entrenando CatBoost final OOF...")
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_cb)):
    X_tr  = X_cb.iloc[tr_idx];  X_val = X_cb.iloc[val_idx]
    y_tr  = y_all.iloc[tr_idx]; y_val = y_all.iloc[val_idx]

    dtrain = cb.Pool(X_tr,  y_tr,  cat_features=cat_indices_cb)
    dval   = cb.Pool(X_val, y_val, cat_features=cat_indices_cb)

    m_cb = cb.CatBoostRegressor(**best_params_cb)
    m_cb.fit(dtrain, eval_set=dval, early_stopping_rounds=50, verbose=False)

    oof_cb[val_idx] = m_cb.predict(X_val)
    test_cb += m_cb.predict(cb.Pool(X_cb_test, cat_features=cat_indices_cb)) / N_SPLITS
    scores_cb.append(mape(y_val.values, oof_cb[val_idx]))
    print(f"  Fold {fold+1} MAPE: {scores_cb[-1]:.2f}%")

print(f"CatBoost OOF MAPE: {np.mean(scores_cb):.2f}% ± {np.std(scores_cb):.2f}%")



In [ ]:
# OOF FINAL — XGBOOST (CORREGIDO)
best_params_xgb = {
    "objective": "reg:absoluteerror",
    "eval_metric": "mae",
    "verbosity": 0,
    "seed": RANDOM_SEED,
    "tree_method": "hist",
    "early_stopping_rounds": 50,
    **study_xgb.best_params
}

oof_xgb   = np.zeros(len(train))
test_xgb  = np.zeros(len(test))
scores_xgb = []

print("\nEntrenando XGBoost final OOF...")
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_xgb)):
    X_tr,  X_val = X_xgb.iloc[tr_idx], X_xgb.iloc[val_idx]
    y_tr,  y_val = y_all.iloc[tr_idx],  y_all.iloc[val_idx]

    m_xgb = xgb.XGBRegressor(**best_params_xgb)

    # El fit ahora queda limpio de ese parámetro
    m_xgb.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    oof_xgb[val_idx] = m_xgb.predict(X_val)
    test_xgb += m_xgb.predict(X_xgb_test) / kf.get_n_splits() # Usamos kf para evitar errores de nombre
    scores_xgb.append(mape(y_val.values, oof_xgb[val_idx]))
    print(f"  Fold {fold+1} MAPE: {scores_xgb[-1]:.2f}%")

print(f"XGBoost OOF MAPE: {np.mean(scores_xgb):.2f}% ± {np.std(scores_xgb):.2f}%")


In [ ]:
# BLENDING ÓPTIMO — BÚSQUEDA EN GRILLA 3 MODELOS
print("\nBuscando pesos óptimos de blend...")

best_mape_blend = 999
best_w = (0.34, 0.33, 0.33)

for w1 in np.arange(0.0, 1.01, 0.05):
    for w2 in np.arange(0.0, 1.01 - w1, 0.05):
        w3 = round(1.0 - w1 - w2, 4)
        if w3 < 0: continue
        blend = w1 * oof_lgbm + w2 * oof_cb + w3 * oof_xgb
        m = mape(y_all.values, blend)
        if m < best_mape_blend:
            best_mape_blend = m
            best_w = (w1, w2, w3)

print(f"Pesos óptimos → LGBM: {best_w[0]:.2f} | CB: {best_w[1]:.2f} | XGB: {best_w[2]:.2f}")
print(f"MAPE blend OOF: {best_mape_blend:.2f}%")

final_oof  = best_w[0] * oof_lgbm  + best_w[1] * oof_cb  + best_w[2] * oof_xgb
final_test = best_w[0] * test_lgbm + best_w[1] * test_cb + best_w[2] * test_xgb

In [ ]:
# MÉTRICAS FINALES
train_price = train["lastSoldPrice_hpi_adjusted"]
pred_price  = np.expm1(final_oof)

# MAPE se calcula sobre y_raw (no winsorizados) para comparar justo
mape_final    = mape(y_raw.values, final_oof)
meanroi_final = mean_roi(y_raw.values, final_oof)

print(f"\n{'='*47}")
print(f"{'RESULTADOS FINALES (BLEND 3 MODELOS)':^47}")
print(f"{'='*47}")
print(f"  R²  (log):   {r2_score(y_raw, final_oof):.4f}")
print(f"  MAE ($):     ${mean_absolute_error(train_price, pred_price):,.0f}")
print(f"  MAPE (%):    {mape_final:.2f}%")
print(f"  Mean ROI(%): {meanroi_final:.2f}%")
print(f"{'='*47}")
print(f"  Baseline anterior: 25.89%")
print(f"  Mejora comparacion a 01_jaime_lgbm_OOF:            {25.89 - mape_final:.2f} pp")

In [ ]:
# FEATURE IMPORTANCE (LGBM)
import matplotlib.pyplot as plt

fi_model = lgb.LGBMRegressor(**best_params_lgbm)
fi_model.fit(X_all, y_all, categorical_feature=CAT_FEATURES,
             callbacks=[lgb.log_evaluation(period=-1)])

fi_df = pd.DataFrame({
    'feature':    FEATURES,
    'importance': fi_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 15 features:")
print(fi_df.head(15).to_string(index=False))

plt.figure(figsize=(10, 9))
top = fi_df.head(25)
plt.barh(top['feature'][::-1], top['importance'][::-1])
plt.title('Top 25 Feature Importances (LightGBM)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# GUARDAR SUBMISSIONS
suffix = f"mape{mape_final:.1f}"

oof_out = pd.DataFrame({
    "zpid":            train["zpid"],
    "predicted_price": np.expm1(final_oof),
})
oof_file = os.path.join(SUBMISSIONS_DRIVE_DIR, f"stack3_oof_{suffix}.csv")
oof_out.to_csv(oof_file, index=False)

test_out = pd.DataFrame({
    "zpid":            test["zpid"],
    "predicted_price": np.expm1(final_test),
})
test_file = os.path.join(SUBMISSIONS_DRIVE_DIR, f"Ronda_2_tabular{suffix}.csv")
test_out.to_csv(test_file, index=False)

print(f"\nOOF guardado:  {oof_file}")
print(f"Test guardado: {test_file}")
print(f"\nMAPE OOF final: {mape_final:.2f}%")